## 一、System Prompt（系统提示词）
System Prompt 是在对话开始前给模型的"工作说明"，定义角色、行为规则和输出格式。它的优先级高于用户消息。

In [2]:
import anthropic
import os
os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"

client = anthropic.Anthropic()

# ── 基础用法 ──────────────────────────────────────────────
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1024,
    # system 定义模型的身份、能力边界、输出规则
    system="""你是一个专注于 AI 领域的技术面试官。
你的职责：
- 只回答与 AI/ML 相关的问题
- 回答简洁，不超过 100 字
- 遇到非技术问题，礼貌拒绝并引导回正题
- 始终用中文回复""",
    messages=[{"role": "user", "content": "什么是 RAG？"}]
)

print(response.content[0].text)

**RAG（检索增强生成）** 是一种将信息检索与大语言模型结合的技术框架。

**核心流程：**
1. **检索**：从外部知识库中找到相关文档
2. **增强**：将检索内容注入提示词
3. **生成**：LLM 基于上下文生成答案

**优势：**
- 减少幻觉
- 知识可实时更新
- 无需重新训练模型

常用于企业知识库问答、文档助手等场景。


System Prompt 的三个核心要素：
|要素 | 作用 | 示例 |
| ------- | ------- | ------- |
|角色定义 | 告诉模型它是谁 |「你是一个专业的代码审查员」|
|行为规则 | 约束输出方式 |「只用中文回复，不超过200字」|
|输出格式 | 规定返回结构 |「始终返回 JSON 格式」

In [3]:
# ── 强制 JSON 输出的 System Prompt ────────────────────────
system_json = """你是一个信息提取助手。
无论用户输入什么，始终以以下 JSON 格式回复，不要有任何多余的文字：
{
  "answer": "回答内容",
  "confidence": 0.0到1.0之间的置信度,
  "source": "依据来源"
}"""

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=256,
    system=system_json,
    messages=[{"role": "user", "content": "Transformer 是什么时候提出的？"}]
)
# 输出：{"answer": "2017年", "confidence": 0.99, "source": "Attention Is All You Need"}

## 二、Few-shot（少样本提示）
在 prompt 里提供几个输入→输出的示例，让模型学会你期望的输出格式和风格，不需要微调。

In [4]:
# ── Zero-shot：不给例子，直接问 ───────────────────────────
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=256,
    messages=[{"role": "user", "content": "判断情感：今天天气真好"}]
)
# 输出不稳定，可能是长段分析

# ── Few-shot：给 3 个例子，锁定输出格式 ──────────────────
few_shot_system = """你是一个情感分析助手。
判断用户输入的情感，只返回：正面/负面/中性

示例：
输入：今天吃到了超好吃的拉面
输出：正面

输入：快递又丢了，气死我了
输出：负面

输入：今天是周三
输出：中性"""

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=16,
    system=few_shot_system,
    messages=[{"role": "user", "content": "这个项目终于上线了！"}]
)
# 输出：正面   ← 格式稳定可控

Few-shot 的两种写法：

In [5]:
# 写法一：放在 system prompt 里（推荐，结构清晰）
system = """...示例放在这里..."""

# 写法二：放在 messages 历史里（模拟真实对话）
messages = [
    {"role": "user", "content": "今天吃到了超好吃的拉面"},
    {"role": "assistant", "content": "正面"},
    {"role": "user", "content": "快递又丢了，气死我了"},
    {"role": "assistant", "content": "负面"},
    {"role": "user", "content": "这个项目终于上线了！"},  # ← 真正要问的
]

几个实用技巧：

* 3 到 5 个例子通常已经足够，太多反而干扰
* 例子要覆盖边界情况，不只给"标准答案"
* 例子的输出格式要和期望完全一致

## 三、Chain-of-Thought（思维链）
让模型先推理再给答案，显著提升复杂问题的准确率。核心原理：把一个大问题拆成多个小步骤，每步都有中间结果，减少跳步出错的概率。

In [6]:
# ── 不用 CoT：直接给答案，容易出错 ──────────────────────
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=64,
    messages=[{
        "role": "user",
        "content": "小明有 3 个苹果，给了小红一半，小红又买了 4 个，小红现在有几个？"
    }]
)
# 可能直接输出错误答案

# ── CoT 方式一：在 prompt 里说「一步一步思考」───────────
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": """小明有 3 个苹果，给了小红一半，小红又买了 4 个，小红现在有几个？
请一步一步思考，展示推理过程，最后给出答案。"""
    }]
)
# 输出：
# 第一步：小明有 3 个苹果，给了小红一半 = 3 ÷ 2 = 1.5，取整为 1 个
# 第二步：小红又买了 4 个
# 第三步：小红总共有 1 + 4 = 5 个
# 答案：5 个

# ── CoT 方式二：Few-shot CoT（给出推理示例）─────────────
cot_system = """解题时，先分析已知条件，再逐步推导，最后给出答案。

示例：
问题：小王有 10 元，买了 3 元的水，还剩多少？
推理：
- 已知：小王有 10 元
- 操作：买水花了 3 元
- 计算：10 - 3 = 7 元
答案：7 元"""

# ── CoT 方式三：XML 标签结构化思维链 ─────────────────────
structured_cot = """请按以下格式回答：
<thinking>
在这里写出你的推理过程
</thinking>
<answer>
在这里写最终答案
</answer>"""

CoT 什么时候用？
|场景 | 是否需要CoT|
|--------|--------|
|数学/逻辑推理|✅ 必须用|
|多步骤决策|✅ 必须用|
|代码调试分析|✅ 建议用|
|简单问答（首都、日期）|❌ 不需要，浪费 token|
|情感分类|❌ 不需要|

## 四、Temperature 控制
Temperature 控制模型输出的随机程度，本质是在 softmax 前对 logits 做缩放。

In [8]:
import anthropic

client = anthropic.Anthropic()

prompt = "用一句话描述春天"

# ── Temperature = 0：完全确定，每次输出相同 ──────────────
# 适合：代码生成、数据提取、分类任务
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=64,
    temperature=0,       # 贪心解码，选概率最高的 token
    messages=[{"role": "user", "content": prompt}]
)
print(f"T=0: {response.content[0].text}")
# 输出固定：春天是万物复苏、百花盛开的季节。

# ── Temperature = 0.7：均衡，默认推荐值 ─────────────────
# 适合：对话、问答、一般文本生成
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=64,
    temperature=0.7,
    messages=[{"role": "user", "content": prompt}]
)
print(f"T=0.7: {response.content[0].text}")

# ── Temperature = 1.0：更有创意，有一定随机性 ────────────
# 适合：头脑风暴、创意写作
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=64,
    temperature=1.0,
    messages=[{"role": "user", "content": prompt}]
)
print(f"T=1.0: {response.content[0].text}")

T=0: 春天是万物复苏、百花争艳、生机勃勃的美好季节。
T=0.7: 春天是万物复苏、百花争艳、生机勃勃的美好季节。
T=1.0: 春天是万物复苏、百花争艳、生机勃勃的美好季节。


In [10]:
import anthropic
client = anthropic.Anthropic()

# 换成这个 prompt，每个选项概率更均衡
prompt = "给我一个 AI 创业公司的名字，只说名字本身，不要解释"

for temp in [0, 0.7, 1.0]:
    # 每个 temperature 重复运行 3 次，观察稳定性
    results = []
    for _ in range(3):
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=32,
            temperature=temp,
            messages=[{"role": "user", "content": prompt}]
        )
        results.append(response.content[0].text.strip())
    print(f"T={temp}: {results}")

# 预期结果：
# T=0:   ['NeuralEdge', 'NeuralEdge', 'NeuralEdge']   ← 完全相同
# T=0.7: ['NeuralEdge', 'MindForge', 'NeuralEdge']    ← 偶尔不同
# T=1.0: ['PixelMind', 'ThinkFlow', 'DeepSpark']      ← 每次不同

T=0: ['**Lumiq**', '**Lumiq**', '**Lumina AI**']
T=0.7: ['**Lumina AI**', '**Lumina AI**', '**Lumiq**']
T=1.0: ['**Lumina AI**', '**Lumiq**', '**Lumina AI**']


不同场景的推荐值：
|场景|Temperature|原因|
|----------|----------|-----------|
|代码生成|0 ~ 0.2|需要正确性，不要随机|
|数据提取 / 分类|0|输出必须稳定一致|
|问答 / 对话|0.5 ~ 0.7|均衡准确和自然|
|创意写作|0.8 ~ 1.0|需要多样性和新颖感|
|头脑风暴|1.0|越随机越好|

Temperature 生效的条件
|条件|Temperature有没有用|
|--------|---------|
|问题有唯一正确答案（2+2=？）|❌ 没用，T=0 和 T=1 都输出 4|
|问题有强势答案（描述春天）|❌ 基本没用，高概率答案碾压问题|
|有多个同等合理答案（起名字、头脑风暴）|✅ 效果明显|
|长文本生成（写故事）|✅ 越往后差异越大，误差累积|

In [13]:
# ✅ 适合观察 temperature 差异的 prompt 类型：

# 1. 创意生成
"用一个意想不到的比喻描述人工智能"

# 2. 开放性续写
"从前有一个程序员，他发现自己的代码开始"

# 3. 多选项头脑风暴
"给 RAG 系统起一个产品名，要简短有力，只说名字"

# 4. 风格化写作
"用李白的风格写一句关于 GPU 的诗"

'用李白的风格写一句关于 GPU 的诗'

In [12]:
# ✅ 只用 temperature
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=256,
    temperature=0.7,       # 只设这一个
    messages=[{"role": "user", "content": "给我一个创业点子"}]
)
print(response.content[0].text)

# ✅ 只用 top_p
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=256,
    top_p=0.9,             # 只设这一个，不传 temperature
    messages=[{"role": "user", "content": "给我一个创业点子"}]
)
print(response.content[0].text)

# 创业点子：**AI驱动的"职场技能速成"平台**

---

## 💡 核心概念
针对职场人士，提供**碎片化、场景化**的技能学习，用AI个性化定制学习路径，帮助用户在**30天内**掌握一项可变现的职场技能。

---

## 🎯 目标用户
- 想转行但不知从何开始的上班族
- 想提升竞争力的应届生
- 想学副业技能的自由职业者

---

## 🔧 核心功能

| 功能 | 说明 |
|------|------|
| AI诊断 | 测评用户现有技能，推荐最适合的学习方向 |
| 微课模块 | 每天15分钟，利用通勤/午休时间学习 |
| 实战
# 创业点子：**AI驱动的"情绪衣橱"穿搭助手**

---

## 💡 核心概念
根据用户**当天情绪、天气、日程和场合**，AI自动从用户已有衣物中推荐最佳穿搭方案。

---

## 🎯 解决的痛点
- 每天"不知道穿什么"浪费大量时间
- 买了很多衣服却总觉得没衣服穿
- 穿搭与场合不匹配的尴尬

---

## 🛠️ 产品功能

| 功能 | 描述 |
|------|------|
| 衣橱数字化 | 拍照上传，AI自动分类标签 |
| 每日穿搭推荐 | 结合天气/日程/心情生成方案 |


temperature vs top_p 怎么选
|参数|机制|适合场景|
|---------|---------|---------|
|temperature|缩放整个概率分布，值越高越随机|大多数场景，直觉更好调|
|top_p|只从累积概率前 p% 的 token 里采样|需要限制「离谱输出」时|

## 五、组合实战：完整的 Prompt 工程模板
把以上四个技巧组合在一起，用于 RAG 问答场景：

In [14]:
import anthropic
import json

client = anthropic.Anthropic()

def rag_answer(question: str, context: str) -> dict:
    """
    结合 System Prompt + CoT + 结构化输出的完整示例
    适用于 RAG 问答场景
    """
    # ① System Prompt：定义角色 + 输出格式
    system = """你是一个严谨的知识问答助手。

规则：
- 只根据提供的上下文回答，不要使用额外知识
- 如果上下文不足以回答，明确说「根据现有信息无法回答」
- 始终返回以下 JSON 格式，不要有任何额外文字：
{
  "answer": "回答内容",
  "reasoning": "推理过程",
  "confidence": 0.0到1.0,
  "has_answer": true或false
}"""

    # ② 用户消息：CoT 引导 + Few-shot 格式示例
    user_message = f"""上下文信息：
{context}

问题：{question}

请先在 reasoning 字段中写出你的推理步骤，再给出最终答案。"""

    # ③ Temperature = 0：问答任务需要稳定输出
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=512,
        temperature=0,
        system=system,
        messages=[{"role": "user", "content": user_message}]
    )

    # 解析 JSON 输出
    try:
        result = json.loads(response.content[0].text)
    except json.JSONDecodeError:
        result = {"answer": response.content[0].text, "confidence": 0.5}

    return result


# 测试
context = """
Transformer 是 Google 在 2017 年提出的深度学习架构，
论文名为《Attention Is All You Need》。
它的核心机制是自注意力（Self-Attention），
支持并行计算，解决了 RNN 长距离依赖的问题。
"""

result = rag_answer("Transformer 是哪年提出的？", context)
print(json.dumps(result, ensure_ascii=False, indent=2))
# 输出：
# {
#   "answer": "2017年",
#   "reasoning": "上下文明确提到 Transformer 是 Google 在 2017 年提出的",
#   "confidence": 0.99,
#   "has_answer": true
# }

{
  "answer": "```json\n{\n  \"answer\": \"Transformer 是 2017 年提出的。\",\n  \"reasoning\": \"在提供的上下文中，明确提到「Transformer 是 Google 在 2017 年提出的深度学习架构」，因此可以直接从原文中提取该信息，得出答案为 2017 年。\",\n  \"confidence\": 1.0,\n  \"has_answer\": true\n}\n```",
  "confidence": 0.5
}


四个技巧的关系总结：